# Module 7: Multi-agent, an orchestrator and specialists

So far you built one agent that does everything. It reads your account, prices resources,
draws diagrams, and makes changes behind an approval gate. That one agent now carries a long
list of tools and a system prompt full of rules for every job. In this module you will split
it into a small team: focused specialist agents, and an orchestrator that routes the work and
calls each specialist like a tool. The running example is still the Akamai Cloud Solutions
Architect.

## Learning objectives
- See why one agent with every tool gets harder for a small model to drive
- Wrap an agent as a tool so another agent can call it
- Build three specialists: documentation, account, and pricing
- Give the documentation specialist real Akamai docs with a local index and a live fetch
- Keep pricing honest with a read-before-reason split between two specialists
- Assemble an orchestrator that routes a request and combines the answers
- Know when not to wrap a tool in an agent

## Prerequisites
- Finished Modules 1 through 6
- The same vLLM endpoint and `LINODE_TOKEN` from the earlier modules
- The local docs index. Build it once before you start:

  ```bash
  python workshop/07_multi_agent/scripts/refresh_docs_index.py
  ```
- The `graphviz` Python package and the `dot` binary for the diagram tool
- About 30 minutes

References: [Strands Agents](https://strandsagents.com) · [Agents as tools](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/multi-agent/agents-as-tools/) · [akamai-cloud-mcp](https://pypi.org/project/akamai-cloud-mcp/) · [Akamai TechDocs](https://techdocs.akamai.com)

## Multi-agent design basics

One agent with twenty tools and a long prompt does two hard jobs at once. It picks the right
tool from a crowded list, and it follows every rule for every domain in one pass. A small model
does both worse as the list grows.

The fix is a team. Give each job to a focused agent with a short prompt and a few tools. Put
one orchestrator in front. The orchestrator does not answer the question itself. It routes each
part of the request to a specialist and combines what they return.

In Strands this is the "agents as tools" pattern. A specialist agent is wrapped in a function
and handed to the orchestrator as just another tool. The orchestrator calls it the same way it
calls `calculator`.

![An orchestrator routes a request to documentation, account, and pricing specialists, then combines their answers](../images/07_multiagent_architecture.png)

## 1. Setup

We use the same Strands and MCP packages from the earlier modules, plus `graphviz` for a small
diagram tool. The documentation specialist reads the local index you built in the prerequisites,
so run this notebook from the `workshop/07_multi_agent` directory.

In [ ]:
%pip install -q strands-agents strands-agents-tools "mcp>=1.0" httpx python-dotenv graphviz

In [ ]:
import os
import re
from pathlib import Path

import httpx
from dotenv import load_dotenv
from graphviz import Digraph
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from strands.tools.mcp import MCPClient
from strands_tools import calculator, think
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()

BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
MODEL_ID = os.getenv("VLLM_MODEL_ID", "Qwen/Qwen2.5-7B-Instruct")
API_KEY = os.getenv("VLLM_API_KEY", "placeholder")
LINODE_TOKEN = os.getenv("LINODE_TOKEN", "")
MCP_DIR = os.getenv("AKAMAI_MCP_DIR", "")  # local checkout; empty -> published package

INDEX_PATH = Path("data/llms.txt")
print("Docs index present:", INDEX_PATH.exists())

## 2. Configure the model

Every agent shares one model object. It holds the endpoint and the HTTP client. The
conversation lives on each agent, not on the model, so one model can back the orchestrator and
all three specialists. One vLLM endpoint, one connection.

In [ ]:
model = OpenAIModel(
    client_args={"api_key": API_KEY, "base_url": BASE_URL},
    model_id=MODEL_ID,
    params={"temperature": 0.2},
)
print("Model:", MODEL_ID)

## 3. The problem: one agent wearing every hat

Start with the agent you already know: one agent, the account tools, the calculator, and a
prompt that holds the account rules and the cost rules together. The MCP server owns a stdio
process, so the agent runs inside the `with` block while the server is alive, the same lifecycle
from Module 2.

Ask it a question that spans two domains: get a real price, then do math on it. It works, but
look at the shape of the problem. One model is choosing from a long tool list and following
every rule in one prompt. We have not even added documentation yet. Adding more to this one
agent is the wrong direction.

In [ ]:
def mcp_server(domains="regions,pricing,compute,lke", max_results=10):
    """Describe how to launch the akamai-cloud-mcp server over stdio."""
    scope = ["--domains", domains, "--max-results", str(max_results)]
    if MCP_DIR:
        command, args = "uv", ["run", "--directory", MCP_DIR, "akamai-cloud-mcp", *scope]
    else:
        command, args = "uvx", ["akamai-cloud-mcp", *scope]
    return StdioServerParameters(command=command, args=args, env={**os.environ, "LINODE_TOKEN": LINODE_TOKEN})


ONE_AGENT_PROMPT = (
    "You are an Akamai Cloud Solutions Architect. Use the read tools for real account data. "
    "Report the monthly and hourly price the tool returns. Use the calculator for all math: "
    "a month is 730 hours, show the per-item cost and the total. Never invent a price."
)

with MCPClient(lambda: stdio_client(mcp_server())) as client:
    account_tools = client.list_tools_sync()
    one_agent = Agent(
        model=model,
        system_prompt=ONE_AGENT_PROMPT,
        tools=account_tools + [calculator, think],
    )
    print("This one agent carries", len(one_agent.tool_names), "tools.\n")
    _ = one_agent(
        "What is the cheapest Akamai Cloud GPU plan, and what does it cost to run 3 of them for a month?"
    )

## 4. The fix: agents as tools

Instead of one agent with every tool, build a few focused agents and let an orchestrator call
them. Each specialist is a normal Strands `Agent` with a short prompt and a couple of tools. You
wrap it in a function decorated with `@tool`. Inside the function you build the specialist and
return its answer as text. To the orchestrator, the specialist is just a tool that takes a
question and returns a string.

```python
@tool
def pricing_agent(task: str) -> str:
    "Do cost math: totals, monthly projections, comparisons."
    specialist = Agent(model=model, system_prompt=PRICING_PROMPT, tools=[calculator, think])
    return str(specialist(task))
```

Strands also has `Swarm` and `Graph` for multi-agent systems. A swarm lets peer agents hand off
to each other. A graph runs a fixed path you define. We use agents as tools here because it is
the same tool-calling you already know from Modules 1 through 3, and the orchestrator decides who
to call. We set `callback_handler=None` on each specialist so only the orchestrator prints.

Before the framework, see the shape with plain Python. Routing is a dict of `{name: function}`:
you call one specialist, then feed its answer to the next. The Strands version is the same
dispatch, only the model picks the specialist instead of you.

In [ ]:
# By hand first: routing is a dict of {name: function}. You call one specialist,
# then feed its answer to the next. No framework, no model, just Python.
def account_stub(question):
    return "The cheapest GPU plan is $0.52 per hour, $350 per month."

def pricing_stub(facts):
    return "Given " + facts + " three of them is about $1050 per month."

SPECIALISTS = {"account": account_stub, "pricing": pricing_stub}

facts = SPECIALISTS["account"]("cheapest GPU plan?")   # look it up
print(SPECIALISTS["pricing"](facts))                   # then do the math

# The @tool version below is the same dispatch, except the model decides which
# specialist to call and when, instead of you hard-coding it.

## 5. A specialist as a tool: documentation

The first specialist answers Akamai product and best-practice questions from the real docs, not
from the model's memory. Akamai publishes an index of every doc page at
`https://techdocs.akamai.com/llms.txt`. It is about 1.8 MB, too big to hand to a small model, so
we keep it local (you built it in the prerequisites) and use it as a table of contents.

The retrieval tool does two things. It searches the local index for the pages that match the
question, then it fetches those few pages live from the net and returns their text. The agent
answers from what it fetched.

In [ ]:
# The Akamai Cloud product docs live under these two paths in the index.
ALLOWED = ("https://techdocs.akamai.com/cloud-computing/", "https://techdocs.akamai.com/linode-api/")
ENTRY = re.compile(r"^- \[(.*?)\]\((https?://[^)]+\.md)\)(?::\s*(.*))?$", re.M)
STOP = {"how", "the", "do", "to", "my", "in", "on", "of", "and", "or", "for", "is", "are",
        "what", "with", "you", "your", "can", "does", "this", "that", "it", "be", "from",
        "use", "get", "set", "any", "all"}


def tokens(s):
    return set(re.findall(r"[a-z0-9]+", s.lower()))


def index_entries():
    """Parse the local index into (title, url, description) rows for the Cloud docs."""
    rows = []
    for m in ENTRY.finditer(INDEX_PATH.read_text(encoding="utf-8")):
        title, url, desc = m.group(1), m.group(2), (m.group(3) or "")
        if url.startswith(ALLOWED):
            rows.append((title, url, desc))
    return rows


def search_docs(question, k=3):
    """Score index rows by whole-word overlap, weighting the title higher."""
    qwords = {w for w in tokens(question) if w not in STOP and len(w) > 2}
    scored = []
    for title, url, desc in index_entries():
        tw, ow = tokens(title), tokens(desc + " " + url)
        score = sum((3 if w in tw else 0) + (1 if w in ow else 0) for w in qwords)
        if score:
            scored.append((score, title, url))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:k]


def clean_page(markdown):
    """Each page repeats an index preamble. Start at the first H1."""
    i = markdown.find("\n# ")
    return (markdown[i + 1:] if i != -1 else markdown).strip()


print("Cloud doc pages in the index:", len(index_entries()))
for score, title, url in search_docs("How do I resize an LKE node pool?"):
    print("  [%d] %s" % (score, title))

In [ ]:
@tool
def docs_lookup(question: str) -> str:
    """Find and read the Akamai Cloud doc pages that answer a question.

    Searches the local docs index, then fetches the top matching pages live from
    techdocs.akamai.com and returns their text with source URLs.

    Args:
        question: An Akamai Cloud product or best-practices question.
    Returns:
        The matching doc pages' text, each with its source URL.
    """
    hits = search_docs(question, 3)
    if not hits:
        return "No matching Akamai Cloud docs page was found in the index."
    parts = []
    for _score, title, url in hits:
        try:
            body = clean_page(httpx.get(url, timeout=20, follow_redirects=True).text)[:2000]
        except Exception as exc:  # noqa: BLE001
            body = "(could not fetch this page: %s)" % exc
        parts.append("## %s\nSource: %s\n\n%s" % (title, url, body))
    return "\n\n---\n\n".join(parts)


DOCS_PROMPT = (
    "You are an Akamai Cloud documentation specialist. Use the docs_lookup tool to ground every "
    "answer in the official docs. Quote what the page says and cite the source URL. If the docs "
    "do not cover it, say so. No em-dashes."
)


@tool
def documentation_agent(question: str) -> str:
    """Answer Akamai Cloud product and best-practice questions from the official docs."""
    specialist = Agent(model=model, system_prompt=DOCS_PROMPT, tools=[docs_lookup], callback_handler=None)
    return str(specialist(question))


# Try the specialist on its own. It searches the index, fetches real pages, and answers.
print(documentation_agent("How do I drain a node pool on an LKE cluster?"))

## 6. Two more specialists: account and pricing

The next two specialists split one job that a single agent often gets wrong: pricing. The trick
is a read-before-reason split.

- The **account specialist** looks things up. It returns the catalog price exactly as the tool
  gives it, hourly and monthly. It never multiplies or totals.
- The **pricing specialist** does the math. It gets the prices and counts in the request and runs
  them through the calculator. It has no account tools, so it cannot look a price up. That is the
  point. A small model cannot hallucinate a price it has no way to fetch.

The orchestrator gets the price from the account specialist first, then hands those numbers to
the pricing specialist. Look up the dollar, then do the math, in two different agents.

In [ ]:
PRICING_PROMPT = (
    "You are a cost analyst for Akamai Cloud. The prices and counts you need are in the request. "
    "Do the math with the calculator. A month is 730 hours. Show the per-item cost and the total, "
    "hourly and monthly. State your assumptions. Never invent a price. If a number is missing, say "
    "which one. No em-dashes."
)

ACCOUNT_PROMPT = (
    "You are an Akamai Cloud account specialist. Use your read tools to report real facts: "
    "inventory, region and GPU availability, and catalog prices exactly as the tools return them "
    "(both hourly and monthly). Do not multiply, total, or recommend. If a value is missing, say "
    "so. No em-dashes."
)


@tool
def pricing_agent(task: str) -> str:
    """Do cost math: totals, monthly projections, comparisons. Give it the prices and counts."""
    specialist = Agent(model=model, system_prompt=PRICING_PROMPT, tools=[calculator, think], callback_handler=None)
    return str(specialist(task))


# The pricing specialist has no account tools. It only computes what it is given.
print(pricing_agent("A GPU plan is $0.52 per hour and $350 per month. What is the monthly cost of 3 of them?"))

The account specialist needs the MCP read tools, and those are only live while the MCP client is
open. So we define it inside the `with` block, where `account_tools` exists, the same way we will
build the orchestrator in the next section.

In [ ]:
with MCPClient(lambda: stdio_client(mcp_server())) as client:
    account_tools = client.list_tools_sync()

    @tool
    def account_agent(question: str) -> str:
        """Read-only Akamai Cloud account facts and catalog prices."""
        specialist = Agent(model=model, system_prompt=ACCOUNT_PROMPT, tools=account_tools, callback_handler=None)
        return str(specialist(question))

    # On its own, the account specialist reports prices and stops. It does not do the math.
    print(account_agent("What is the cheapest Akamai Cloud GPU plan? Give the hourly and monthly price."))

## 7. The orchestrator

Now put one orchestrator in front. Its tools are the three specialists, plus a small diagram
tool. Its prompt is a router: it says which specialist handles what, and it enforces the
read-before-reason order for pricing. It does not answer account or pricing questions itself.

The orchestrator and the account specialist both need the MCP tools, so everything runs inside
one `with` block. Ask one question that spans all three specialists and watch it route: get the
price from the account specialist, hand it to the pricing specialist, and pull the docs from the
documentation specialist.

In [ ]:
@tool
def draw_diagram(title: str, components: str, connections: str) -> str:
    """Render an architecture diagram and return the image path.

    Args:
        title: a short title for the diagram.
        components: comma-separated component names, e.g. "Control plane, Worker 1, Worker 2".
        connections: comma-separated edges as A->B, e.g. "Control plane->Worker 1".
    """
    g = Digraph(format="png")
    g.attr(rankdir="LR")
    for c in [c.strip() for c in components.split(",") if c.strip()]:
        g.node(c, c)
    for pair in connections.split(","):
        if "->" in pair:
            a, b = pair.split("->", 1)
            g.edge(a.strip(), b.strip())
    Path("out").mkdir(exist_ok=True)
    path = g.render("out/design", cleanup=True)
    return "Diagram '%s' saved to %s" % (title, path)


ROUTER_PROMPT = (
    "You are an Akamai Cloud Solutions Architect that coordinates specialists. You do not answer "
    "account or pricing questions from your own knowledge. Route each part of the request to the "
    "right specialist, then combine their answers.\n\n"
    "Your specialists (tools):\n"
    "- documentation_agent: Akamai product facts and best practices from the official docs.\n"
    "- account_agent: real account data and catalog prices (look-ups only).\n"
    "- pricing_agent: cost math, totals, monthly projections, recommendations.\n"
    "- draw_diagram: render an architecture diagram.\n\n"
    "Rules:\n"
    "- For any cost total, FIRST get the per-unit price from account_agent, THEN give those numbers "
    "to pricing_agent. Never ask pricing_agent to look up a price.\n"
    "- Use one specialist per step, in order. Then write one combined answer. No em-dashes."
)

with MCPClient(lambda: stdio_client(mcp_server())) as client:
    account_tools = client.list_tools_sync()

    @tool
    def account_agent(question: str) -> str:
        """Read-only Akamai Cloud account facts and catalog prices."""
        specialist = Agent(model=model, system_prompt=ACCOUNT_PROMPT, tools=account_tools, callback_handler=None)
        return str(specialist(question))

    orchestrator = Agent(
        model=model,
        system_prompt=ROUTER_PROMPT,
        tools=[documentation_agent, account_agent, pricing_agent, draw_diagram, think],
        callback_handler=None,
    )

    result = orchestrator(
        "What do Akamai Cloud GPU plans cost? Price running 3 of the cheapest GPU plan for a month, "
        "and point me to the official docs on LKE node pools."
    )

    # The route it took: the specialist tools the orchestrator called, in order.
    route = [b["toolUse"]["name"] for m in orchestrator.messages
             for b in (m.get("content") or []) if isinstance(b, dict) and "toolUse" in b]
    print("Route:", " -> ".join(route), "\n")
    print(result)

The route is the lesson. The orchestrator called `account_agent` for the price, handed those
numbers to `pricing_agent` for the math, and `documentation_agent` for the docs, then wrote one
answer. It never priced anything itself. Each specialist saw a short prompt and a few tools, which
is what keeps a small model on task.

## 8. When not to wrap a tool in an agent

Notice `draw_diagram` is a plain tool on the orchestrator, not its own agent. That is on purpose.

Wrapping a tool in an agent buys you one thing: a model that can reason about how to use it,
across several steps, with its own prompt. The documentation specialist earns that. It decides
what to search, reads pages, and writes a grounded answer. `draw_diagram` does not. It takes a
title and some nodes and renders them. There is nothing to reason about, so a whole extra agent
loop would just add a round-trip and a chance to get it wrong.

The rule: wrap a tool in an agent when the job needs judgment over several steps. Keep it a plain
tool when one call does the work. The deterministic diagram tools from Module 5 are the same case.
They stay tools.

## 9. Where writes and the approval gate live

The orchestrator is the only agent that should hold the write tools and the approval gate from
Module 3. There is a concrete reason, not just tidiness. The approval grant travels in the
agent's `invocation_state`, set on the top-level call. A specialist built inside a `@tool`
function is a fresh agent that does not inherit that state, so a write routed through a specialist
could never receive its approval. Keep every write, and the one `ApprovalGate`, on the
orchestrator. Specialists only read, reason, and render, so they need no gate.

Sessions from Module 4 live on the orchestrator too. The specialists are built fresh on each call
and do not remember earlier turns. The orchestrator holds the conversation.

## Try it yourself

The honest tradeoff of a team of agents is more model calls. One request becomes the orchestrator
plus a full loop for each specialist it routes to, all on one vLLM endpoint. Measure it.

1. **Pull your eval cases.** Take three or four cases from your Module 6 evals.
2. **Race the two designs.** Run each against the single agent from section 3 and against the orchestrator.
3. **Compare routing and time.** Check which routed where, and how long each took.

One catch to watch. The Module 6 harness reads `agent.messages` to see which tools were used. With
the orchestrator, the specialist's tool calls live on the specialist's messages, not the
orchestrator's. So a check for `expect_tools=["linode_list_..."]` will look like it failed even
when the agent did the right thing. Fixing that, by looking inside each specialist, is the real
lesson: in a multi-agent system your logging has to follow the call tree.

In [ ]:
import time

# Race the flat single agent against the orchestrator on one query. Swap in your own
# Module 6 cases, and look at which route each took.
QUERY = "What regions does Akamai Cloud offer? Name three."

with MCPClient(lambda: stdio_client(mcp_server())) as client:
    account_tools = client.list_tools_sync()

    @tool
    def account_agent(question: str) -> str:
        """Read-only Akamai Cloud account facts and catalog prices."""
        spec = Agent(model=model, system_prompt=ACCOUNT_PROMPT, tools=account_tools, callback_handler=None)
        return str(spec(question))

    flat = Agent(model=model, system_prompt=ONE_AGENT_PROMPT,
                 tools=account_tools + [calculator, think], callback_handler=None)
    orch = Agent(model=model, system_prompt=ROUTER_PROMPT,
                 tools=[documentation_agent, account_agent, pricing_agent, draw_diagram, think],
                 callback_handler=None)

    t = time.time(); _ = flat(QUERY); flat_s = time.time() - t
    t = time.time(); _ = orch(QUERY); orch_s = time.time() - t
    print("flat agent: %.1fs    orchestrator: %.1fs" % (flat_s, orch_s))

## Summary

- One agent with every tool makes a small model pick from a long list and follow every rule at
  once. A team of focused agents is easier to drive and to extend.
- "Agents as tools" wraps a specialist agent in a `@tool` function. The orchestrator calls it like
  any other tool and combines the results.
- Split pricing into read-before-reason: the account specialist looks the price up, the pricing
  specialist does the math and has no way to look anything up.
- Wrap a tool in an agent only when the job needs judgment over several steps. Keep `draw_diagram`
  and the Module 5 diagram tools as plain tools.
- Writes, the approval gate, and the session stay on the orchestrator. The grant only reaches the
  top-level call.

## Next

**Module 8: Deploy to LKE.** The agent only runs on your laptop. Next you will package this
orchestrator and ship it to Akamai Cloud on a Kubernetes cluster behind a NodeBalancer.